# exchanger 온도 기반 진단 — 왜 모델을 못 만들었나 (시도 총정리)

**목표였던 것**: 온도만 보고 "exchanger 릴레이가 이번에 실제로 작동했는가"를
판별하는 모델. **결론: 못 만들었다.** 이 노트북은 왜 안 되는지를, 시도한
순서 그대로, 실제 데이터로 다시 증명한다.

## 미리 보는 이유들

| # | 이유 | 한 줄 요약 |
|---|---|---|
| 1 | 전용 센서 없음 | `Scale_Out___TT_EX_*` 존재하지만 값이 전부 0(미사용) |
| 2 | 선택 편향 + **문턱 제어** | `Para_DM_H_ExSet`(23.0/21.0℃) 문턱에서 켜진다. 제어기가 온도를 붙들어 매서 **측정하려는 신호를 스스로 지운다** |
| 3 | 효과가 잡음 대비 작음 | feeding 대비 신호가 10~20배 약하다(Cohen's d) |
| 4 | 토글이 너무 빠름 | 일부 탱크는 "꺼진 지 15초"만에 또 켜져서 깨끗한 전후비교 자체가 불가 |
| 5 | 창 길이 오염 | 탱크 진동주기보다 창이 길면 부호까지 뒤집힌다(P5 사례) |
| 6 | 정화하면 표본이 줄어듦 | 오염 제거하면 신호는 좋아져도 남는 샘플이 너무 적어짐(탱크별로 갈림) |
| 7 | 상태값(T)이 신호를 삼킴 | SHAP 분해에서 온도 자체의 비선형성이 액추에이터 신호를 가림 |
| 8 | **가짜 고장 주입에서 전부 실패** | 좋아 보였던 방법들도 검증하면 못 잡는다 — 결정타 |

> ### 🔄 2026-09-14 추가 — 결론이 바뀐 부분이 있다
> 위 1~8 은 전부 **"온도 변화량으로 이벤트 하나하나를 판정"** 하려던 시도다.
> 그 목표는 여전히 불가능하다. 그런데 이유 2 를 정정하면서 **제어기가 온도를
> 문턱에 고정시킨다**는 걸 알게 됐고, 그렇다면 정보는 온도가 아니라
> **"얼마나 오래 켜져 있어야 했는가(가동률)"** 에 있다. **이유 9 에서 그 방법이
> 가짜고장 주입 검증을 통과한다** — exchanger 방법 중 처음이다.
> 대신 판정 단위가 "이벤트 하나"가 아니라 **"30분 구간"** 으로 바뀐다.

**feeding과 정확히 대칭이다.** feeding은 결과(수위)가 직접 계측되고, 효과가
잡음의 수십 배라 한 건만 봐도 확실했다. exchanger는 결과를 온도로 간접
추정해야 하는데, 그 온도가 다른 3개 요인(heat/feed/자연손실)에도 동시에
영향받고, 효과 자체도 작다 — 근본적으로 다른 난이도다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.getcwd())

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG

BASE_DIR = os.path.dirname(os.getcwd())
SAVE_DIR = os.path.join(BASE_DIR, "extracted_csv")

EXCLUDE_HOURS = (0, 7)
RUNNING_WINDOW_MIN = 30
MIN_EVENT_SEC = 5

pd.set_option("display.width", 220)
rng = np.random.default_rng(0)


def load(tank_key):
    cfg = TANKS[tank_key]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True,
                      usecols=["Time", "source", WAGON_TAG, cfg["temp"], cfg["exchanger"],
                              cfg["heat"], cfg["cool"], cfg["feeding"]]).sort_index()
    d = pd.DataFrame({"T": raw[cfg["temp"]], "exch": raw[cfg["exchanger"]],
                      "heat": raw[cfg["heat"]], "cool": raw[cfg["cool"]], "feed": raw[cfg["feeding"]],
                      "wagon": raw[WAGON_TAG], "diag": (raw["source"] == "DIAGNOSTIC").astype(float)})
    w = d["wagon"].resample("1min").last().ffill()
    run = (w.diff().abs() > 0).rolling(RUNNING_WINDOW_MIN, min_periods=1).max()
    d["running"] = run.reindex(d.index, method="ffill").fillna(0)
    h = d.index.hour
    d["ok"] = (((h < EXCLUDE_HOURS[0]) | (h >= EXCLUDE_HOURS[1])) & (d["diag"] == 0) & (d["running"] == 1))
    return d


def events(d, col, minsec=MIN_EVENT_SEC):
    s = (d[col] == 1) & d["ok"]
    grp = (s & ~s.shift(1, fill_value=False)).cumsum()[s]
    return [(idx.iloc[0], idx.iloc[-1]) for _, idx in d.index[s].to_series().groupby(grp)
            if (idx.iloc[-1] - idx.iloc[0]).total_seconds() >= minsec]


def slope(seg):
    if len(seg) < 3:
        return np.nan
    x = (seg.index - seg.index[0]).total_seconds().values
    return np.polyfit(x, seg.values, 1)[0] * 60 if x[-1] > 0 else np.nan


TARGET_TANKS = list(TANKS.keys())
print("대상 탱크:", TARGET_TANKS)

## 이유 1 — 전용 센서가 없다

exchanger 전용 온도 태그(`Scale_Out___TT_EX_*`)가 실제로 존재한다. 있다면
feeding 의 수위처럼 "결과가 직접 찍히는 계측"이 될 수 있었다. 그런데 실측하면
**값이 전부 0** 이다 — PLC 에 설정만 있고 실제로는 안 쓰는 채널이다.

In [ ]:
ex_path = sorted(glob.glob(os.path.join(SAVE_DIR, "EX_Temp_Probe_*.csv")))
if ex_path:
    df_ex = pd.read_csv(ex_path[-1], index_col="Time", parse_dates=True)
    ex_cols = [c for c in df_ex.columns if c.startswith("Scale_Out___TT_EX_")]
    print(f"{ex_path[-1]} 에서 확인:")
    for col in ex_cols:
        v = df_ex[col]
        print(f"  {col:<22} 고유값={sorted(v.dropna().unique())[:5]}  전부0? {bool((v==0).all())}")
else:
    print("EX_Temp_Probe_*.csv 없음 — Custom_Tag_Extractor.py 로 Scale_Out___TT_EX_* 를 다시 받아서 확인 가능")

## 이유 2 — 선택 편향, 그리고 **실제 제어 법칙** (2026-09-14 정정)

> ### ⚠️ 이 절에는 두 가지 오류가 있었다. 둘 다 아래에서 정정한다.
> **(가) 단위 오류.** 아래 표의 "차이" 열은 PLC **원시 단위(0.1℃)** 인데 ℃ 로
> 표기했었다 — 실제 차이는 **10배 작다.** 4.9~27.6 이 아니라 **0.49~2.76℃** 다.
> **(나) 근거 오류.** "온도가 높을 때 켜진다"고 뭉뚱그렸는데, 실제 제어는
> **명시적 설정값 태그와의 비교**다.

### 실제 제어 법칙 — `Para_DM_H_ExSet_*`

DB 에 8개 탱크 전부 설정값 태그가 있다(`Para_DM_H_ExSet_P1` … `_I3`). 값은
**상수**이고 P1·P3·P4·P5·I1·I3 = 230, P2·I2 = 210 (0.1℃ 단위 → 23.0℃ / 21.0℃).
아래 셀에서 ON 순간의 실제 온도를 재보면 **표준편차가 0.07~0.19℃ 밖에 안 된다**
— 탱크 전체 온도 변동폭(σ 1.0~2.9℃)의 **10~30분의 1**이다.

### 정정해도 결론은 같고, 오히려 더 정확해진다

"온도가 높아서 켜졌다"를 이제 **"T 가 고정 문턱을 넘어서 켜졌다"** 로 정확히
말할 수 있다. 그리고 이게 **온도 기울기 방법이 왜 그렇게 약했는지**를 설명한다:

> **이건 히스테리시스가 있는 on-off(bang-bang) 제어기다.** 제어기의 목적 자체가
> 온도를 문턱에 **붙들어 매는 것**이라, 정상 작동 중에는 온도가 거의 안 움직인다.
> ON 온도와 OFF 온도가 사실상 같다(아래 표). **제어기가 우리가 측정하려는
> 바로 그 신호를 능동적으로 지워버리고 있다.**

그래서 "온도 변화량"은 정보가 거의 없다. **대신 정보는 "얼마나 오래/자주
켜져 있어야 했는가"(가동률) 로 옮겨간다** — 이유 9 에서 이걸 쓴다.

In [ ]:
EXSET = {"P1": 23.0, "P2": 21.0, "P3": 23.0, "P4": 23.0,
         "P5": 23.0, "I1": 23.0, "I2": 21.0, "I3": 23.0}   # Para_DM_H_ExSet_* ÷ 10

rows = []
for tk in TANKS:
    d = load(tk)
    T = d["T"] / 10.0                      # ← PLC 원시단위(0.1℃) → ℃. 예전엔 이걸 빠뜨렸다
    ok = d["ok"]
    on_T, off_T = T[(d["exch"] == 1) & ok], T[(d["exch"] == 0) & ok]
    trans = T[(d["exch"] == 1) & (d["exch"].shift(1) == 0) & ok].dropna()
    if len(on_T) < 100 or len(trans) < 20:
        continue
    rows.append({"탱크": tk, "ExSet(℃)": EXSET[tk],
                 "exch=1 평균T": round(float(on_T.mean()), 2),
                 "exch=0 평균T": round(float(off_T.mean()), 2),
                 "차이(℃)": round(float(on_T.mean() - off_T.mean()), 2),
                 "ON순간 T중앙": round(float(trans.median()), 2),
                 "ON순간 T표준편차": round(float(trans.std()), 3),
                 "탱크 전체 T표준편차": round(float(T[ok].std()), 3)})
res2 = pd.DataFrame(rows)
print(res2.to_string(index=False))
print()
print("차이(℃) 열: 예전 노트북은 이 값을 10배로(원시단위 그대로) 적어 4.9~27.6 이라 했다.")
print("ON순간 T표준편차 ≪ 탱크 전체 T표준편차  →  거의 고정된 온도에서 켜진다(문턱 제어).")

## 이유 3 — 효과 크기가 잡음 대비 작다 (feeding과 정면 비교)

같은 잣대(Cohen's d = 평균차이 ÷ 표준편차)로 feeding 과 exchanger 를 나란히
잰다. **d 가 클수록 "한 건만 봐도 확실하다"** 는 뜻이다.

- feeding: ON 직후 수위 변화 vs OFF 구간의 수위 변화
- exchanger: ON 직후 60초 온도 변화 vs 무작위 대조 시점의 온도 변화

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled = np.sqrt((a.var() * (len(a) - 1) + b.var() * (len(b) - 1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else np.nan


rows = []
for tk in TARGET_TANKS:
    cfg = TANKS[tk]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tk}_Temp_vs_Factors_*.csv")))[-1]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True,
                      usecols=["Time", cfg["level"], cfg["feeding"]])
    lv, feed = raw[cfg["level"]], raw[cfg["feeding"]]
    d60 = lv.shift(-12) - lv   # 대략 60초 근방(비균일이라 근사)
    feed_on = d60[feed == 1].dropna()
    feed_off = d60[feed == 0].dropna()

    d = load(tk)
    on_dT, ctl_dT = [], []
    for t0, t1 in events(d, "exch")[:400]:
        seg = d.loc[t0: t0 + pd.Timedelta(seconds=60), "T"].dropna()
        if len(seg) >= 3:
            on_dT.append(seg.iloc[-1] - seg.iloc[0])
    idx = d.index[d["ok"] & (d["exch"] == 0)]
    for i in rng.choice(len(idx), size=min(1500, len(idx)), replace=False):
        t = idx[i]
        seg = d.loc[t: t + pd.Timedelta(seconds=60), "T"].dropna()
        if len(seg) >= 3:
            ctl_dT.append(seg.iloc[-1] - seg.iloc[0])
        if len(ctl_dT) >= 400:
            break

    if len(feed_on) < 30 or len(on_dT) < 30:
        continue
    rows.append({"탱크": tk,
                 "feeding_d": round(cohens_d(feed_on, feed_off), 2),
                 "exchanger_d": round(cohens_d(on_dT, ctl_dT), 2)})

cmp_d = pd.DataFrame(rows)
cmp_d["배수(feeding/exchanger)"] = (cmp_d["feeding_d"].abs() / cmp_d["exchanger_d"].abs()).round(1)
cmp_d

## 이유 4 — 토글이 너무 빨라서 "깨끗한" 전후 비교 자체가 안 되는 탱크가 있다

exchanger 효과를 재려면 "켜지기 직전"이 오염 안 된 상태여야 한다. 그런데
꺼진 지 얼마 안 돼 또 켜지는 탱크는, 기준(직전) 구간 자체가 방금 전
exchanger 효과의 잔향에 걸쳐 있다.

In [ ]:
rows = []
for tk in TARGET_TANKS:
    d = load(tk)
    ev = events(d, "exch")
    if len(ev) < 20:
        continue
    gaps = [(ev[i][0] - ev[i-1][1]).total_seconds() for i in range(1, len(ev))]
    gaps = pd.Series(gaps)
    overlap_120 = (gaps < 120).mean() * 100
    rows.append({"탱크": tk, "이벤트수": len(ev), "OFF간격중앙(초)": round(gaps.median(), 0),
                 "120초창과겹침비율%": round(overlap_120, 0)})
pd.DataFrame(rows).sort_values("120초창과겹침비율%", ascending=False)

**겹침 비율이 높을수록** "직전 구간이 깨끗한" 이벤트가 적다는 뜻이다.
I1·P3처럼 65~71%가 겹치면, 애초에 쓸 수 있는 "깨끗한" 이벤트가 전체의
1/3도 안 남는다.

## 이유 5 — 창 길이가 탱크 진동주기보다 길면 부호까지 뒤집힌다 (P5 사례)

P5 는 heat/cool 이 4분 주기로 도는데, 처음엔 모든 탱크에 똑같이 120/60초
창을 썼다. 창이 주기의 76%나 되니, 앞뒤 창이 톱니파의 서로 다른 위상에
걸려서 exchanger 효과가 진동에 묻혔다.

In [ ]:
def heat_period(d):
    s = d["heat"][d["ok"]]
    on = s.index[(s == 1) & (s.shift(1) == 0)]
    return float(pd.Series(on).diff().dt.total_seconds().median()) if len(on) >= 5 else np.nan


d5 = load("P5")
period = heat_period(d5)
print(f"P5 heat/cool 진동주기 중앙값: {period:.0f}초 ({period/60:.1f}분)")
print()
print("=== 창 길이(전/후초)를 줄여가며 재본 P5 의 exchanger 효과 ===")
for pre, post in [(120, 60), (60, 30), (30, 15), (20, 10)]:
    on_ds, ctl_ds = [], []
    for t0, t1 in events(d5, "exch")[:400]:
        a = slope(d5.loc[t0 - pd.Timedelta(seconds=pre): t0, "T"].dropna())
        b = slope(d5.loc[t0: t0 + pd.Timedelta(seconds=post), "T"].dropna())
        if not (np.isnan(a) or np.isnan(b)):
            on_ds.append(b - a)
    idx = d5.index[d5["ok"] & (d5["exch"] == 0)]
    for i in rng.choice(len(idx), size=min(1200, len(idx)), replace=False):
        t = idx[i]
        if d5.loc[t - pd.Timedelta(seconds=300): t, "exch"].max() > 0:
            continue
        a = slope(d5.loc[t - pd.Timedelta(seconds=pre): t, "T"].dropna())
        b = slope(d5.loc[t: t + pd.Timedelta(seconds=post), "T"].dropna())
        if not (np.isnan(a) or np.isnan(b)):
            ctl_ds.append(b - a)
        if len(ctl_ds) >= 300:
            break
    diff = np.mean(on_ds) - np.mean(ctl_ds)
    print(f"  창 {pre}/{post}초 (주기의 {(pre+post)/period*100:.0f}%): "
          f"평균차이 = {diff:+.2f}   {'(부호 반대!)' if diff > 0 else ''}")

## 이유 6 — 오염을 없애면(정화) 신호는 좋아져도 표본이 너무 줄어드는 탱크가 있다

기준 구간을 한 칸 떼고(갭), 직전 이벤트와 안 겹치는 것만 남기면(정화)
신호대잡음비(SNR = 평균|효과| / 표준편차)가 좋아지는 탱크도 있고, 표본이
너무 깎여서 오히려 나빠지는 탱크도 있다.

In [ ]:
def cycle_window(d):
    period = heat_period(d)
    if np.isnan(period):
        return 120, 60
    total = period * 0.15
    pre = int(np.clip(total * 2 / 3, 20, 120))
    return pre, pre // 2


def snr_variants(tk):
    d = load(tk)
    pre, post = cycle_window(d)
    ev = events(d, "exch")
    if len(ev) < 30:
        return None
    A, B, C = [], [], []
    prev_end = None
    for t0, t1 in ev:
        b = slope(d.loc[t0: t0 + pd.Timedelta(seconds=post), "T"].dropna())
        a1 = slope(d.loc[t0 - pd.Timedelta(seconds=pre): t0, "T"].dropna())
        a2 = slope(d.loc[t0 - pd.Timedelta(seconds=pre+post): t0 - pd.Timedelta(seconds=post), "T"].dropna())
        clean = prev_end is None or (t0 - prev_end).total_seconds() >= (pre + post)
        prev_end = t1
        if np.isnan(b):
            continue
        if not np.isnan(a1):
            A.append(b - a1)
        if not np.isnan(a2):
            B.append(b - a2)
            if clean:
                C.append(b - a2)

    def snr(v):
        v = np.asarray(v)
        return abs(v.mean()) / v.std() if len(v) > 20 and v.std() > 0 else np.nan

    return {"탱크": tk, "①현재(n)": f"{snr(A):.3f}({len(A)})",
            "②갭(n)": f"{snr(B):.3f}({len(B)})", "③갭+정화(n)": f"{snr(C):.3f}({len(C)})"}


rows = [r for tk in TARGET_TANKS if (r := snr_variants(tk))]
pd.DataFrame(rows)

## 이유 7 — SHAP 분해에서는 온도 자체(T)의 비선형성이 액추에이터 신호를 통째로 가린다

heat/exchanger/feed 중 어느 게 "온도 예측이 크게 틀린" 원인인지 SHAP 으로
분해해보면, **T(상태값) 가 8개 탱크 전부에서 80~100% 비율로 1등**을 차지한다.
T 는 액추에이터가 아니다 — 선형모델이 T 자체의 비선형 반응(뜨거울수록 냉각이
가속되는 효과)을 못 잡아서 생기는 모델링 한계일 뿐인데, 이게 진짜 원인
신호를 덮어버린다. (자세한 도출 과정은 `Temp_Residual_Attribution.ipynb` 참고.
숫자만 요약하면 아래와 같다.)

In [ ]:
t_dominance = pd.DataFrame({
    "탱크": ["P1", "P2", "P3", "P4", "P5", "I1", "I2", "I3"],
    "T가_주요차이요인_비율%": [100, 100, 100, 100, 100, 80, 100, 100],
})
print("(2026-09-14 시점 실측, Temp_Residual_Attribution.ipynb 섹션 4에서 재현 가능)")
t_dominance

## 이유 8 (결정타) — 가짜 고장을 실제로 주입하면, 좋아 보였던 방법도 전부 실패한다

이유 6 에서 SNR 이 좋았던 I2(1.454)·I3(1.539)·P1(0.677) 을 대상으로, feeding
모델에 썼던 것과 **완전히 같은 틀**(이동 기준선 + z중심보정 + CUSUM + 학습/
홀드아웃 임계값 산정)을 그대로 적용하고, 마지막 20개 이벤트의 반응을
학습구간에서 재표본한 "정상 반응"으로 대체해 완전정지를 흉내낸다.

In [ ]:
import Exchanger_Slope_Validation as ESV

val = ESV.main()

**임계값 바로 아래서 봉우리를 찍고 다시 내려간다.** P1 은 CUSUM 이 6.31 까지
올라갔다가(임계값 6.64) 못 넘고 내려왔고, I3 는 8.48 까지만 가고(임계값
12.22) 한참 못 미쳤다. **원인은 이동 기준선 자체다** — 완전정지가 20개
이벤트나 계속되면, 그 30개짜리 이동 기준선이 "0 이 새로운 정상"이라고
학습해버려서 CUSUM 이 스스로 가라앉는다. SNR 이 아무리 좋아도, **적응형
기준선을 쓰는 한 이 함정을 피할 수 없다.**

I2 는 정화 후 이벤트가 40개뿐이라 학습/검증(70/30) 분할조차 못 해서
"표본부족"으로 나온다 — 애초에 검증할 수조차 없다.

---

# 이유 9 — 가동률(duty) 잔차: **한때 된다고 봤으나 철회한다** (2026-09-14)

> ## ❌ 이 절의 첫 결론("된다")은 틀렸다. 왜 틀렸는지를 아래에 남긴다.
> 처음엔 R² 0.88~0.97, 홀드아웃 오경보 0 으로 나와 "exchanger 방법 중 처음으로
> 검증 통과"라고 적었다. **실수가 두 가지 있었다** — 이유 10 에서 실측으로
> 보인다. 틀린 과정을 지우지 않는 이유는, 이게 **앞의 1~8번과 똑같은 함정**
> (그럴듯해 보이다가 제대로 검증하면 무너짐)의 아홉 번째 사례이기 때문이다.

## 왜 이 방향을 진작 안 봤나

이유 2 를 정정하면서 알게 된 사실 때문이다. exchanger 는 **`Para_DM_H_ExSet`
문턱(23.0℃ / 21.0℃)에서 켜지고 꺼지는 on-off 제어기**다. 제어기의 목적이
온도를 그 문턱에 붙들어 매는 것이라, **정상 작동 중에는 온도가 거의 안 움직인다.**

1~8 번의 모든 시도는 "온도가 얼마나 변했나"를 쟀다. 그런데 **제어기가 그 변화를
없애려고 일하고 있다.** 온도를 아무리 정교하게 봐도 정보가 거기 없다.

**정보는 다른 곳에 있다.** 냉각 능력이 떨어지면 같은 온도를 유지하기 위해
**더 오래 켜져 있어야 한다.** 즉 신호는 온도가 아니라 **가동률(duty cycle,
구간 중 ON 이었던 시간 비율)** 이다.

```
가동률 = (30분 구간 중 exchanger 가 ON 이었던 시간) / 30분
```

## 교란 요인은 다른 신호로 뺀다 (← "다른 신호를 빼면 되지 않나"의 답)

가동률은 열화 말고도 **부하**에 따라 변한다 — 더 뜨거운 액체가 들어오거나
히터가 많이 돌면 exchanger 도 많이 돌아야 한다. 그래서 같은 구간의
`heat`, `cool`, `feed`, `T` 로 가동률을 회귀하고 **잔차**를 본다.

```
가동률 ≈ f(heat, cool, feed, T)   ← 부하가 설명하는 부분
잔차   = 실제 가동률 − 예측 가동률  ← 부하로 설명 안 되는 부분 = 진단 신호
```

**이게 1~8번과 결정적으로 다른 점**: 1~8번은 "이벤트 하나"를 판정하려 했는데
매번 잡음에 묻혔다. 여기서는 **30분 구간 전체를 하나의 관측**으로 본다 —
수백 번의 ON/OFF 가 평균되면서 잡음이 √n 로 줄어든다.

In [ ]:
from sklearn.linear_model import LinearRegression

WIN, BW, TRAIN_FRAC = "30min", 20, 0.7
FEATS = ["heat", "cool", "feed", "T"]

def duty_table(tk, win=WIN):
    """30분 구간별 가동률 + 부하 특징."""
    d = load(tk)
    d = d[d["ok"]]
    g = d.resample(win).agg({"exch": "mean", "heat": "mean", "cool": "mean",
                             "feed": "mean", "T": "mean"}).dropna()
    return g[g["exch"] > 0]

rows = []
for tk in TANKS:
    g = duty_table(tk)
    if len(g) < 40:
        continue
    X, y = g[FEATS].values, g["exch"].values
    m = LinearRegression().fit(X, y)
    resid = y - m.predict(X)
    rows.append({"탱크": tk, "구간수": len(g),
                 "평균가동률": round(float(y.mean()), 3),
                 "R²(부하로 설명)": round(m.score(X, y), 3),
                 "잔차/가동률 %": round(float(resid.std() / y.mean() * 100), 1)})
print("=== 가동률을 부하로 설명하고 남은 잔차 ===")
print(pd.DataFrame(rows).to_string(index=False))
print()
print("'잔차/가동률 %' = 한 구간만 보고 감지 가능한 최소 열화율(1σ). 작을수록 좋다.")

## 검증 — 앞의 모든 방법을 떨어뜨린 그 틀을 그대로 적용

SNR 이 좋아 보였던 방법들이 **가짜 고장 주입에서 전부 실패**했다(이유 8).
그러니 이 방법도 같은 시험을 통과해야만 한다. 완전히 동일하게:

- 앞 70% 로만 회귀를 적합하고(뒤 기간을 안 보고), 이동 기준선 + CUSUM
- 임계값은 `Feeding_Model.threshold_by_arl` (목표 ARL 500)
- **뒤 30% 에서 실제 오경보를 센다**
- 마지막 20구간에 가짜 고장 주입 — 냉각 능력이 `sev` 만큼 떨어지면 같은 열을
  빼내려고 가동률이 `1/(1−sev)` 배로 올라간다고 모형화

> **주입 모형이 가정이라는 점을 명시한다.** 실제 고장 데이터가 없어서
> "열화 → 가동률 상승" 이라는 물리 모형을 쓴다. on-off 제어기에서는 타당한
> 가정이지만 **관측된 사실은 아니다.**

In [ ]:
import Feeding_Model as FM

def validate_duty(tk):
    g = duty_table(tk)
    if len(g) < BW + 25:
        return {"탱크": tk, "상태": "표본부족"}
    n_tr = int(len(g) * TRAIN_FRAC)
    X, y = g[FEATS].values, g["exch"].values
    m = LinearRegression().fit(X[:n_tr], y[:n_tr])       # 학습 구간에서만 적합
    v = -(y - m.predict(X))                              # 고장=가동률↑ → 부호 반전
    z_tr = FM.rolling_z(v[:n_tr], window=BW)
    bias = float(np.nanmean(z_tr))
    thr, _ = FM.threshold_by_arl(z_tr - bias)
    cs = FM.cusum_down(FM.rolling_z(v, window=BW) - bias)
    fa = int((cs[n_tr:] >= thr).sum())

    r = {"탱크": tk, "구간": len(g), "학습": n_tr,
         "R²": round(m.score(X[:n_tr], y[:n_tr]), 3),
         "임계값": round(thr, 1), "홀드아웃오경보": fa,
         "오경보율%": round(fa / (len(g) - n_tr) * 100, 1)}
    for sev in (0.10, 0.25, 0.50):
        yi = y.copy()
        yi[-20:] = np.minimum(yi[-20:] / (1 - sev), 1.0)
        vi = -(yi - m.predict(X))
        csi = FM.cusum_down(FM.rolling_z(vi, window=BW) - bias)[-20:]
        hit = int(np.argmax(csi >= thr)) + 1 if (csi >= thr).any() else None
        r[f"{int(sev*100)}%열화"] = hit           # 몇 번째 30분 구간에 잡히나
    return r

print("=== 가동률 잔차 방법 — 가짜고장 주입 검증 (숫자 = 몇 번째 30분 구간에 검출) ===")
print(pd.DataFrame([validate_duty(tk) for tk in TANKS]).to_string(index=False))

## 이유 9 결과 해석

**P5·P4·P2·I3 는 쓸 수 있다.** 홀드아웃 오경보 0 이면서 25% 열화를
1~4구간(30분~2시간) 안에 잡는다. 특히 **P5 는 10% 열화를 첫 구간(30분)에**
잡는다 — 부하 설명력이 R² 0.97 로 높아 잔차가 3.2% 밖에 안 되기 때문이다.

**P1·I1 은 부분적이다.** 부하 설명력이 R² 0.43~0.63 으로 낮아 잔차가
13~34% 다. 50% 이상 크게 망가져야 잡힌다.

**I2 는 못 쓴다.** 홀드아웃 오경보가 2건(5.9%) 나고 검출도 11구간 이상 걸린다.
가동률 평균이 0.053 으로 워낙 낮아(거의 안 켜짐) 잔차 비율이 38% 다.

| 탱크 | R² | 잔차% | 오경보 | 25% 열화 검출 | 판정 |
|---|---|---|---|---|---|
| P5 | 0.97 | 3.2 | 0 | **30분** | ✅ 우수 |
| P4 | 0.88 | 3.5 | 0 | **1시간** | ✅ 우수 |
| P2 | 0.88 | 5.3 | 0 | 1.5시간 | ✅ 양호 |
| I3 | 0.89 | 10.2 | 0 | 2시간 | ✅ 양호 |
| P3 | 0.48 | 13.5 | 0 | 3시간 | ⚠️ 제한적 |
| P1 | 0.63 | 13.3 | 0 | 미검출(50%만) | ⚠️ 제한적 |
| I1 | 0.43 | 34.2 | 0 | 미검출(50%만) | ⚠️ 제한적 |
| I2 | 0.89 | 38.0 | **2** | 6시간 | ❌ 불가 |

## 한계 — 과장하지 않기 위해 명시한다

1. **판정 단위가 이벤트가 아니라 30분 구간이다.** "이번 ON 이 실패했다"는
   여전히 말 못 한다. "이 30분 동안 평소보다 더 일했다"까지다. 원래 목표였던
   이벤트 단위 진단은 **여전히 불가능**하다.
2. **주입 모형이 가정이다** (열화 → 가동률 1/(1−sev) 배). 실제 고장 기록이
   없어서 검증할 방법이 없다.
3. **`T` 를 설명변수로 쓴다.** 그런데 exchanger 가 약해지면 T 도 조금 오른다 —
   즉 T 는 결과이기도 하다(사후변수). 회귀가 신호 일부를 흡수했을 수 있어
   **실제 성능은 위 표보다 나쁠 수 있다.**
4. **4.8일 / 130구간뿐이다.** `heat` 를 빼도 결과가 동일한데(`cool` 과
   거의 완전 공선), 이런 건 표본이 늘면 달라질 수 있다.
5. **완전 정지는 이 방법이 필요 없다** — 가동률이 1.0 에 붙어버려 눈으로 보인다.
   이 방법의 가치는 **부분 열화(10~25%)** 구간에 있다.

---

# 이유 10 — 제어 법칙을 정확히 알고 난 뒤의 **최종 결론** (2026-09-14)

## 현장에서 받은 제어 법칙

```
exchanger ON  =  (TK_Temp_PV_탱크 > Para_DM_H_ExSet_XX)  AND  (Pump_Run_YY == 1)
```

**설정값 번호가 탱크 번호와 다르다** (P2→ExSet_P3, P3→ExSet_P5, P5→ExSet_P2/P7,
I3→ExSet_I2/I5/I7). 그리고 **펌프 조건**이 있다 — 펌프는 전체의 60% 만 돌고,
꺼져 있으면 온도가 아무리 높아도 exchanger 는 안 켜진다.

## 확정된 두 가지 사실

| 사실 | 의미 |
|---|---|
| `... Exchanger SOL` 은 **PLC 출력 코일**이다 | 밸브가 실제로 움직였다는 신호가 **아니다.** "명령했는데 안 켜졌다"는 **정의상 관측 불가** |
| `TT_EX_*` 온도계는 **실물이 없다** | 태그만 있고 값이 0 인 게 아니라 센서 자체가 없다 |

## 남은 유일한 경로, 그리고 그게 막히는 지점

exchanger 가 물리적으로 망가졌을 때 데이터에 남는 흔적은 **온도밖에 없다.**
냉각이 약해지면 온도가 설정값 위에 더 오래 머물고, 그래서 **명령이 더 오래
켜져 있게 된다.** 즉 "초과 시간"은 출력 코일만 있어도 물리적 의미가 있다 —
여기까지는 맞다.

**막히는 곳은 "그게 고장 때문인지 부하 때문인지"다.** 초과 시간은 뜨거운 원료가
많이 들어와도 늘어난다. 부하를 빼야 하는데 **부하를 재는 독립적인 계측이 없다.**
아래 셀들이 이걸 실측으로 보인다.

In [ ]:
RULES = {"P1": [("P1","P1"), ("P6","P6")], "I1": [("I1","I1"), ("I6","I6")],
         "P2": [("P3","P3")], "I2": [("I3","I3"), ("I4","I4")], "P4": [("P4","P4")],
         "P3": [("P5","P5")], "I3": [("I2","I2"), ("I5","I5"), ("I7","I7")],
         "P5": [("P2","P1"), ("P7","P6")]}

dl = pd.read_csv(os.path.join(SAVE_DIR, "ExchLogic_2026-09-14.csv"),
                 index_col=0, parse_dates=True).sort_index()
dl = dl[~dl.index.duplicated(keep="last")].ffill()
_w = dl[WAGON_TAG].resample("1min").last().ffill()
_r = (_w.diff().abs() > 0).rolling(30, min_periods=1).max()
dl["running"] = _r.reindex(dl.index, method="ffill").fillna(0)
dl["wagon_rate"] = _w.diff().abs().reindex(dl.index, method="ffill").fillna(0)
_h = dl.index.hour
dl = dl[(_h >= 7) & (dl["running"] == 1) & (dl["source"] != "DIAGNOSTIC")]


def demand_pump(tk):
    """제어 법칙대로 '요구' 와 '펌프가동' 을 만든다."""
    dem = pd.Series(False, index=dl.index)
    pmp = pd.Series(False, index=dl.index)
    for ei, pi in RULES[tk]:
        dem |= (dl[f"TK_Temp_PV_{tk}"] > dl[f"Para_DM_H_ExSet_{ei}"]) & (dl[f"Pump_Run_{pi}"] == 1)
        pmp |= (dl[f"Pump_Run_{pi}"] == 1)
    return dem, pmp


rows = []
for tk in RULES:
    act = dl[TANKS[tk]["exchanger"]].astype(int)
    dem, _ = demand_pump(tk)
    dem = dem.astype(int)
    rows.append({"탱크": tk, "규칙수": len(RULES[tk]),
                 "요구=1 %": round(float(dem.mean() * 100), 1),
                 "실제ON %": round(float(act.mean() * 100), 1),
                 "일치율 %": round(float((act == dem).mean() * 100), 1)})
print("=== 제어 법칙 재현 검증 ===")
print(pd.DataFrame(rows).to_string(index=False))

## 이유 9 가 왜 틀렸나 — 실수 두 가지

### 실수 ① 순환논리 — 목표변수를 목표변수로 설명했다

이유 9 에서 설명변수에 **평균 온도 T** 를 넣었다. 그런데 목표변수인 가동률도
**T 에서 파생된다**(T > 설정값인 시간의 비율). 자기 자신으로 자기를 설명한
셈이라 R² 가 부풀려졌다. **T 를 빼고 진짜 외생 부하만 쓰면 R² 가 무너진다.**

### 실수 ② 분할을 한 번만 봤다

70/30 한 번만 잘라서 "오경보 0" 이라고 했다. **분할 지점을 4군데로 바꿔보면
오경보가 최대 30% 까지 나온다.** 한 번의 분할은 운이었다.

In [ ]:
from sklearn.linear_model import LinearRegression

aux = {}
for tk, cfg in TANKS.items():
    _p = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tk}_Temp_vs_Factors_*.csv")))[-1]
    _a = pd.read_csv(_p, index_col="Time", parse_dates=True,
                     usecols=["Time", cfg["heat"], cfg["cool"], cfg["feeding"]]).sort_index()
    _a.columns = ["heat", "cool", "feed"]
    aux[tk] = _a[~_a.index.duplicated(keep="last")]

BW = 20
SETS = [(["heat", "cool", "feed", "T", "pump"], "T포함 (이유9 방식)"),
        (["feed", "pump", "wagon_rate"], "외생 부하만 (정직한 버전)")]

rows = []
for tk in RULES:
    over, pump = demand_pump(tk)
    base = pd.DataFrame({"over": over.astype(float), "T": dl[f"TK_Temp_PV_{tk}"] / 10.0,
                         "pump": pump.astype(float), "wagon_rate": dl["wagon_rate"]})
    base = base.join(aux[tk].reindex(dl.index, method="ffill"))
    g = base.resample("30min").mean().dropna()
    g = g[g["pump"] > 0.1]
    if len(g) < 60:
        continue
    y = (g["over"] / g["pump"]).values
    for feats, label in SETS:
        X = g[feats].values
        fas = []
        for frac in (0.5, 0.6, 0.7, 0.8):          # <- 분할을 네 군데로
            n_tr = int(len(g) * frac)
            m = LinearRegression().fit(X[:n_tr], y[:n_tr])
            v = -(y - m.predict(X))
            z_tr = FM.rolling_z(v[:n_tr], window=BW)
            bias = float(np.nanmean(z_tr))
            thr, _ = FM.threshold_by_arl(z_tr - bias)
            cs = FM.cusum_down(FM.rolling_z(v, window=BW) - bias)
            fas.append(round(int((cs[n_tr:] >= thr).sum()) / (len(g) - n_tr) * 100, 1))
        rows.append({"탱크": tk, "설명변수": label,
                     "R2": round(LinearRegression().fit(X, y).score(X, y), 3),
                     "오경보율% (분할 4회)": fas, "최악": max(fas)})

out10 = pd.DataFrame(rows)
for _lab in [s[1] for s in SETS]:
    print()
    print(f"=== {_lab} ===")
    print(out10[out10["설명변수"] == _lab].drop(columns="설명변수").to_string(index=False))

### 결과 해석 — 두 실수가 모두 확인된다

**R² 붕괴** (T 를 빼면):

| 탱크 | T 포함 | 외생만 | |
|---|---|---|---|
| P4 | 0.898 | **0.169** | 설명력의 81% 가 순환논리였다 |
| P2 | 0.844 | **0.108** | |
| I2 | 0.662 | **0.066** | |
| P1 | 0.534 | **0.067** | |

**오경보 폭발** (분할을 바꾸면): P2 최대 29.6%, P5 25.4%, I3 11.9%.
**오경보율이 25% 인 감시기는 쓸 수 없다.**

## 최종 결론

| 목표 | 가능한가 | 근거 |
|---|---|---|
| 이벤트 하나 판정 | ❌ **불가** | 이유 1~8 |
| 부분 열화(10~30%) 감시 | ❌ **불가** | 이유 10 — 부하를 뺄 독립 계측이 없음 |
| **완전 정지 감지** | ✅ **가능 (통계 모델 불필요)** | 아래 |

**부분 열화가 왜 원천적으로 막히는지**가 이제 명확하다. 신호는 "부하 대비
얼마나 더 일했나"인데, **부하를 재는 독립적인 계측이 이 라인에 없다.**
외생 변수만으로는 초과 시간의 7~19% 밖에 설명 못 한다(P5·I3 제외). 나머지
80% 이상이 설명 안 된 채 잡음으로 남아서 25% 열화 신호가 그 안에 묻힌다.

---

## 쓸 수 있는 것 — 통계 없이 되는 단순 규칙

완전 정지가 나면 **냉각이 없으니 온도가 설정값 아래로 내려올 수 없다.**
그러면 "펌프가 도는데 설정값 위에 연속으로 머문 시간"이 무한정 늘어난다.
정상 상태 baseline 을 재보면 이 값이 아주 작아서 **임계 하나만 그으면 된다.**

In [ ]:
rows = []
for tk in RULES:
    over, _ = demand_pump(tk)
    grp = (over != over.shift()).cumsum()[over]
    durs = dl.index.to_series()[over].groupby(grp).agg(
        lambda s: (s.iloc[-1] - s.iloc[0]).total_seconds() / 60)
    durs = durs.values if len(durs) else np.array([0.0])
    rows.append({"탱크": tk, "연속초과 중앙(분)": round(float(np.median(durs)), 1),
                 "95%(분)": round(float(np.percentile(durs, 95)), 1),
                 "최장(분)": round(float(durs.max()), 1),
                 "20분↑ 횟수": int((durs >= 20).sum()),
                 "30분↑ 횟수": int((durs >= 30).sum())})
print("=== 정상 baseline: 펌프 가동 중 설정값 위에 연속으로 머문 시간 (5일) ===")
print(pd.DataFrame(rows).to_string(index=False))

### 규칙

```
펌프가 도는 동안 온도가 설정값 위에 20분 연속으로 머물면  →  exchanger 확인요망
```

정상 5일간 최장이 **2.0~8.4분**이고 **20분을 넘은 적이 0회**다. 여유가 2.4~10배라
오경보 없이 완전 정지를 잡는다. 학습도, 임계값 산정도, 파라미터도 필요 없다.

**한계를 분명히 한다**: 이건 "완전히 죽었다"만 잡는다. 부분 열화는 못 잡고,
원인이 exchanger 인지 냉매 공급인지 펌프인지도 구분 못 한다. 그리고 그 정도로
온도가 오르면 **기존 고온 알람이 먼저 울릴 가능성이 높다** — 즉 이 규칙의
실질 가치는 "새로운 감지"보다 **"원인을 exchanger 쪽으로 좁혀주는 것"** 에 가깝다.

## 여기서 더 가려면 — 계측 추가 외에 방법이 없다

| 추가 계측 | 무엇이 가능해지나 |
|---|---|
| exchanger 냉매 **입출구 온도차** | 열교환량 직접 계산 → 부분 열화 즉시 감지 |
| 냉매 **유량계** | 위와 동일 + 밸브 막힘 구분 |
| 밸브 **개도 피드백 / 리밋 스위치** | "명령했는데 안 움직였다" 이벤트 단위 판정 |

이 중 **입출구 온도차가 가장 싸고 효과가 크다** (온도계 2개).

## 종합표 — 시도한 방법 전체

| 방법 | 핵심 지표 | 최고 성적 | 검증 결과 |
|---|---|---|---|
| ARX 계수 기반 검출비 | 검출비 | P5 0.72~2.5 | 자기상관 무시한 과대평가로 확인, 폐기 |
| 단순 전후 Δ기울기 | Cohen's d | 최대 0.68 | 개별 이벤트 판별 불가 |
| ARX 잔차(조건통제) | Cohen's d | 최대 0.64 | 여전히 부족 |
| 전용 센서(`TT_EX_*`) | - | - | **값이 전부 0, 사용 불가** |
| 창 길이 고정 → 주기비례 | 부호 정합성 | P5 부호 반전 해결 | 필요 조건일 뿐, 충분 조건 아님 |
| 갭+정화 SNR | SNR(평균/표준편차) | I3 1.54, I2 1.45 | **가짜고장 주입 시 실패** (아래) |
| SHAP 기여도 분해(T 포함) | 주요요인 비율 | - | T 가 80~100% 독점 → 무의미 |
| SHAP 기여도 분해(T 제외) | 쏠림비율 | P1·P5 100% | 가짜고장 주입 시 tank별로 13~100% 요동, 대조군 편향 발견 |
| **갭+정화 + CUSUM + 가짜고장 검증** | 완전정지 검출 | **전부 미검출** | P1 6.31/6.64, I3 8.48/12.22 (임계값 못 넘음) |
| 가동률 잔차 (T 포함) | 25% 열화 검출 | R² 0.88~0.97 | ❌ **철회** — 순환논리 + 분할 1회만 |
| 가동률 잔차 (외생부하만) | 25% 열화 검출 | R² 0.07~0.19 | ❌ 부하를 뺄 독립 계측이 없어 불가 |
| **연속 초과시간 규칙** | 완전 정지 | 정상 최장 8.4분 vs 임계 20분 | ✅ **통과** — 통계 모델 불필요 |

## 결론

**8가지 이유가 서로 다른 층위에서 겹쳐서 막고 있다:**
1~2번은 **데이터 자체의 한계**(계측 없음, 제어로직에 의한 인과 역전),
3번은 **물리적 신호 크기의 한계**(feeding 대비 10~20배 약함),
4~6번은 **통계적 처리로 부분 개선은 되지만 완전히 못 없애는 잡음**,
7번은 **분석 도구(선형/SHAP)의 한계**, 그리고 8번이 **최종 확인 도장**이다
— 신호대잡음비가 좋아 보여도, 지속되는 고장을 적응형 기준선으로 잡으려는
접근 자체가 구조적으로 어렵다.

### 2026-09-14 정정된 결론

**"이벤트 하나를 온도로 판정한다"는 목표는 여전히 불가능하다** — 1~8번 그대로다.
근본 원인이 이제 더 정확해졌다: **문턱 제어기가 온도를 고정시켜서, 우리가
측정하려는 신호를 스스로 지우고 있다**(이유 2 정정).

**가동률로 목표를 바꿔봤지만 그것도 안 된다**(이유 9 → 이유 10 에서 철회).
`SOL` 이 출력 코일이고 `TT_EX` 실물이 없다는 게 확정되면서 남은 경로는
"부하 대비 얼마나 더 일했나" 하나였는데 — **부하를 재는 독립 계측이 없다.**

| | 이벤트 단위 | 30분 구간 단위 |
|---|---|---|
| 온도 변화량 | ❌ 불가 (이유 1~8) | ❌ 불가 |
| 가동률/초과시간 잔차 | (해당 없음) | ❌ 불가 (이유 10) |
| **연속 초과시간 규칙** | (해당 없음) | ✅ **완전 정지만 가능** |

**최종 권고**
1. **지금 할 것**: "펌프 가동 중 설정값 위 20분 연속" 규칙 — 통계 없이 즉시 적용
2. **다음 할 것**: exchanger 냉매 **입출구 온도계 2개** — 가장 싸고 효과가 크다.
   이게 붙으면 부분 열화 감지까지 바로 갈 수 있다
3. 그 전까지 exchanger 는 **"완전히 죽었는지"만** 말할 수 있다